In [1]:

import heapq

# Find coordinates of a value in a 2D state grid
def get_coordinates(state, value):
    for r in range(3):
        for c in range(3):
            if state[r][c] == value:
                return r, c
    return None

# Heuristic Function: Manhattan Distance based on dynamic goal positions
def calculate_manhattan(state, goal_positions):
    distance = 0
    for r in range(3):
        for c in range(3):
            val = state[r][c]
            if val != 0:  # Exclude the blank tile from the distance calculation
                goal_r, goal_c = goal_positions[val]
                distance += abs(r - goal_r) + abs(c - goal_c)
    return distance

# Get all valid neighbor states by moving the blank tile (0)
def get_neighbors(state):
    neighbors = []
    r, c = get_coordinates(state, 0)

    # Define possible moves: Up, Down, Left, Right
    moves = {
        'Up': (r - 1, c),
        'Down': (r + 1, c),
        'Left': (r, c - 1),
        'Right': (r, c + 1)
    }

    for action, (nr, nc) in moves.items():
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_state = [list(row) for row in state]
            new_state[r][c], new_state[nr][nc] = new_state[nr][nc], new_state[r][c]
            neighbors.append((tuple(tuple(row) for row in new_state), action))

    return neighbors

# Core A* Search Algorithm
def a_star_search(initial_state, goal_state):
    # Precompute goal positions dynamically
    goal_positions = {goal_state[r][c]: (r, c) for r in range(3) for c in range(3)}

    initial_h = calculate_manhattan(initial_state, goal_positions)

    # Priority Queue elements: (f_score, g_score, current_state, path_taken)
    # path_taken stores: (action, state, g_score, h_score, f_score)
    frontier = []
    heapq.heappush(frontier, (initial_h, 0, initial_state, []))

    visited = set()

    while frontier:
        f, g, current_state, path = heapq.heappop(frontier)

        # Goal Test
        if current_state == goal_state:
            return path, g, initial_h

        if current_state in visited:
            continue
        visited.add(current_state)

        # Generate children nodes
        for neighbor, action in get_neighbors(current_state):
            if neighbor not in visited:
                next_g = g + 1
                next_h = calculate_manhattan(neighbor, goal_positions)
                next_f = next_g + next_h

                next_path = path + [(action, neighbor, next_g, next_h, next_f)]
                heapq.heappush(frontier, (next_f, next_g, neighbor, next_path))

    return None, None, None

# Helper function to print a 3x3 board nicely
def print_board(state):
    for row in state:
        print(" ".join(str(x) if x != 0 else "_" for x in row))

# Helper function to prompt user for board state input
def input_state(state_name):
    print(f"\nEnter the {state_name} state row by row (use 0 for blank space '_'):")
    state_list = []
    for i in range(3):
        row = list(map(int, input(f"Enter row {i+1} separated by spaces: ").split()))
        if len(row) != 3:
            raise ValueError("Each row must have exactly 3 numbers.")
        state_list.append(row)
    return tuple(tuple(row) for row in state_list)

# Main function
def main():
    print("--- 8-Puzzle A* Solver (Custom Initial & Goal States) ---")

    try:
        initial_state = input_state("INITIAL")
        goal_state = input_state("FINAL / GOAL")

        print("\nInitial State:")
        print_board(initial_state)
        print("Final State:")
        print_board(goal_state)

        print("\nRunning A* Search...")
        path, total_cost, initial_h = a_star_search(initial_state, goal_state)

        if path is not None:
            print(f"\nSuccess! Goal reached in {total_cost} moves.\n")

            # Print the metrics for the starting root node
            print("Step 0: Initial State")
            print_board(initial_state)
            print(f"Level (g) = 0 | h value = {initial_h} | f value = {initial_h}\n")
            print("-" * 40)

            # Print metrics for each successive step taken
            for idx, (move, state, g, h, f) in enumerate(path, 1):
                print(f"Step {idx}: Move {move}")
                print_board(state)
                print(f"Level (g) = {g} | h value = {h} | f value = {f}")
                print("-" * 40)
        else:
            print("Failed to find a solution.")

    except ValueError as e:
        print(f"\nInput Error: {e}. Please ensure inputs consist of 0-8 with no duplicates.")

if __name__ == "__main__":
    main()



--- 8-Puzzle A* Solver (Custom Initial & Goal States) ---

Enter the INITIAL state row by row (use 0 for blank space '_'):
Enter row 1 separated by spaces: 2 8 3
Enter row 2 separated by spaces: 1 6 4
Enter row 3 separated by spaces: 7 0 5

Enter the FINAL / GOAL state row by row (use 0 for blank space '_'):
Enter row 1 separated by spaces: 1 2 3 
Enter row 2 separated by spaces: 8 0 4
Enter row 3 separated by spaces: 7 6 5

Initial State:
2 8 3
1 6 4
7 _ 5
Final State:
1 2 3
8 _ 4
7 6 5

Running A* Search...

Success! Goal reached in 5 moves.

Step 0: Initial State
2 8 3
1 6 4
7 _ 5
Level (g) = 0 | h value = 5 | f value = 5

----------------------------------------
Step 1: Move Up
2 8 3
1 _ 4
7 6 5
Level (g) = 1 | h value = 4 | f value = 5
----------------------------------------
Step 2: Move Up
2 _ 3
1 8 4
7 6 5
Level (g) = 2 | h value = 3 | f value = 5
----------------------------------------
Step 3: Move Left
_ 2 3
1 8 4
7 6 5
Level (g) = 3 | h value = 2 | f value = 5
-------------